# 214 — Per-K cluster centroid backfill (non-rawds)

For every run with `cluster_labels_by_k.csv` (KMeans + HC k_range runs)
this walks each K in the sweep and writes:

    cluster_centroids/k_{K}/cluster_{NN}.png

This mirrors the last cell of `212_raw_downsampled_clustering.ipynb` —
which already does it for `rawds`. After this notebook runs, MOBA's K
slider will update chip thumbnails for raw / blob / minus101 / hg
runs too (instead of always showing the saved best-K image via the flat
fallback `cluster_centroids/cluster_NN.png`).

**Skipped automatically:**
* rawds runs (212 already owns them)
* runs without `cluster_labels_by_k.csv` (single-K runs, e.g. `n_clusters` instead of `k_range`)
* runs without `X_train.npy` (older runs from before `fit_and_save` saved features)
* per-K subdirs that already have a full set of PNGs (idempotent re-runs)

**Nothing else is touched** — existing flat-path PNGs (`cluster_centroids/cluster_NN.png`),
the `cluster_centroids/ranked/` dir written by 213, and all other artifacts are left alone.

## 0 — Imports + paths

In [5]:
import os, sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))
from functions import lf_cluster_run as R

CLUSTERING_DIR = Path(R.DEFAULT_OUTPUTS_ROOT)
INDEX_PATH = CLUSTERING_DIR / 'index.json'

assert INDEX_PATH.exists(), f'No {INDEX_PATH} — run a clustering notebook first.'
with open(INDEX_PATH) as f:
    INDEX = json.load(f)
print(f'{len(INDEX["runs"])} runs in index.json')

61 runs in index.json


## 1 — Config

In [6]:
# Feature sets to backfill. 'rawds' is intentionally excluded — 212
# already populates its k_{K}/ subdirs from the 3D X_3d_ds it builds
# in-memory, with vlim=5.0 (matching the dB scale of the original ERSP).
FEATURE_SETS = ['raw', 'hg', "rawds"]

# Re-render even if a k_{K}/ subdir already exists with the right count of PNGs?
FORCE_REGEN = True

# Rendering style (matches 212's rawds backfill convention: square 2×2,
# aspect='auto', bwr, no axes, edge-to-edge).
FIGSIZE = (2, 2)
DPI     = 100
CMAP    = 'bwr'
ASPECT  = 'auto'

# 2D reshape per feature_set. Values from feature_schema.json across the
# existing runs: raw / minus101 are flat 129×300 ERSP; blob is a 48-vector;
# hg is a 300-bin time series. The 1D ones fall through to (1, n_features)
# so the 2×2 figure renders them as a horizontal stripe.
CENTROID_SHAPE = {
    'raw':      (129, 300),
    'minus101': (129, 300),
    # blob, hg => (1, n_features) at runtime
}

print(f'FEATURE_SETS={FEATURE_SETS}  FORCE_REGEN={FORCE_REGEN}')
print(f'CENTROID_SHAPE={CENTROID_SHAPE}')

FEATURE_SETS=['raw', 'hg', 'rawds']  FORCE_REGEN=True
CENTROID_SHAPE={'raw': (129, 300), 'minus101': (129, 300)}


## 2 — Helpers

In [7]:
def _reshape_centroid(mean_vec, feature_set):
    """Reshape a flat cluster-mean vector to the natural 2D image for plotting.
    HG is handled separately (line plot) — this helper just returns the 1D
    vector reshaped to (1, n_features) so other 1D feature sets render as
    a horizontal stripe."""
    mean_vec = np.asarray(mean_vec).ravel()
    shape = CENTROID_SHAPE.get(feature_set)
    if shape is None or shape[0] * shape[1] != mean_vec.size:
        return mean_vec.reshape(1, -1)
    return mean_vec.reshape(shape)


# Fixed y-limits for the HG centroid line plot. Matches cfg.hg_vmin /
# cfg.hg_vmax (±6.5 dB) so every HG centroid + sample chip shares the same
# y-axis scale as the other ERSP feature sets. The previous dynamic
# ymax-based scaling made every HG centroid look "full-range" regardless
# of how peaky it actually was, which masked across-cluster amplitude
# differences.
HG_YLIM = (-6.5, 6.5)


def _save_one_centroid_png(out_path, mean_vec, feature_set, vlim):
    """
    Render a single cluster centroid.

    Rendering style depends on the feature_set:
      * 'hg'   — time-series line plot of mean high-gamma amplitude.
                 Black mean line + fill_between (red = positive, blue =
                 negative) + faint zero baseline, FIXED ±6.5 dB y-limits
                 so HG centroids are comparable across all K cuts and
                 across all HG runs.
      * other  — square 2x2 figure heatmap with cmap='bwr', aspect='auto'.
                 (Existing convention matched to 212's rawds backfill so
                 chip thumbnails read consistently across non-time-series
                 feature sets.)
    """
    mean_vec = np.asarray(mean_vec).ravel()
    fig, ax = plt.subplots(figsize=FIGSIZE)

    if feature_set == 'hg':
        # HG = 1D high-gamma envelope over time. Treat it as a signal,
        # not an image. Negative excursions get blue shading, positive get
        # red, and the mean line is solid black so the trace stays
        # readable. Fixed y-limits at ±6.5 dB (HG_YLIM above) so amplitude
        # scale is comparable across centroids + per-sample HG views.
        t = np.arange(mean_vec.size)
        ax.fill_between(t, 0, mean_vec, where=(mean_vec > 0),
                        color='#d65a5a', alpha=0.55, interpolate=True, linewidth=0)
        ax.fill_between(t, 0, mean_vec, where=(mean_vec < 0),
                        color='#5a7ed6', alpha=0.55, interpolate=True, linewidth=0)
        ax.plot(t, mean_vec, color='black', lw=1.0)
        ax.axhline(0, color='#999', lw=0.35, alpha=0.7)
        ax.set_ylim(*HG_YLIM)
        ax.set_xlim(0, mean_vec.size - 1)
    else:
        img = _reshape_centroid(mean_vec, feature_set)
        ax.imshow(img, aspect=ASPECT, cmap=CMAP,
                  vmin=-vlim, vmax=+vlim, interpolation='nearest')

    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    fig.savefig(out_path, dpi=DPI, bbox_inches='tight', pad_inches=0)
    plt.close(fig)


def _is_subdir_complete(out_dir, n_expected):
    """True if `out_dir` exists and already has >= n_expected cluster_*.png files."""
    if not out_dir.exists():
        return False
    n_have = sum(1 for _ in out_dir.glob('cluster_*.png'))
    return n_have >= n_expected


def backfill_run(run, *, force=False, verbose=True):
    """
    Walk every K in the run's cluster_labels_by_k.csv, generate per-cluster
    centroid PNGs under cluster_centroids/k_{K}/cluster_NN.png.

    Returns (n_K_processed, n_K_skipped, n_pngs_written) for the run.
    """
    run_dir     = CLUSTERING_DIR / run['path']
    feature_set = run['feature_set']

    X_path   = run_dir / 'X_train.npy'
    lbk_path = run_dir / 'cluster_labels_by_k.csv'
    if not X_path.exists():
        if verbose: print(f'  [skip] X_train.npy missing')
        return (0, 0, 0)
    if not lbk_path.exists():
        if verbose: print(f'  [skip] no cluster_labels_by_k.csv (single-K run)')
        return (0, 0, 0)

    X = np.load(X_path)
    lbk = pd.read_csv(lbk_path)

    ks = []
    for col in lbk.columns:
        if col.startswith('k_'):
            try:
                ks.append(int(col[2:]))
            except ValueError:
                pass
    ks.sort()

    # Shared scale: 99th-percentile abs(X). Keeps PNGs comparable within run.
    vlim = float(np.percentile(np.abs(X), 99)) or 1.0

    n_proc = n_skip = n_pngs = 0
    for k in ks:
        labels = lbk[f'k_{k}'].to_numpy()
        uniq = sorted(int(c) for c in np.unique(labels) if c >= 0)
        out_dir = run_dir / 'cluster_centroids' / f'k_{k}'

        if not force and _is_subdir_complete(out_dir, len(uniq)):
            n_skip += 1
            continue
        out_dir.mkdir(parents=True, exist_ok=True)

        for c in uniq:
            idx = np.where(labels == c)[0]
            if idx.size == 0:
                continue
            mean_vec = X[idx].mean(axis=0)
            _save_one_centroid_png(
                out_dir / f'cluster_{c:02d}.png',
                mean_vec, feature_set, vlim,
            )
            n_pngs += 1
        n_proc += 1
        if verbose:
            print(f'  K={k}: wrote {len(uniq)} PNGs -> cluster_centroids/k_{k}/')

    # Free X explicitly — raw is ~238 MB per run.
    del X
    return n_proc, n_skip, n_pngs

## 3 — Main loop

Iterate every run in `index.json` whose `feature_set` is in `FEATURE_SETS`
(rawds is excluded — 212 owns it). For each run with a `cluster_labels_by_k.csv`,
walk every K in the sweep and write per-cluster centroid PNGs.

In [8]:
total_proc = total_skip = total_pngs = 0
for run in INDEX['runs']:
    if run['feature_set'] not in FEATURE_SETS:
        continue
    print(f'\n=== {run["path"]} ({run["feature_set"]}) ===')
    try:
        n_proc, n_skip, n_pngs = backfill_run(run, force=FORCE_REGEN, verbose=True)
    except Exception as e:
        print(f'  [ERROR] {type(e).__name__}: {e}')
        import traceback; traceback.print_exc()
        continue
    total_proc += n_proc
    total_skip += n_skip
    total_pngs += n_pngs

print(f'\nDone. K-cuts processed: {total_proc} · skipped (already complete): {total_skip} · PNGs written: {total_pngs}')


=== hierarchical/hg/runs/20260521_160447 (hg) ===
  [skip] X_train.npy missing

=== hierarchical/hg/runs/20260522_205603 (hg) ===
  [skip] X_train.npy missing

=== hierarchical/hg/runs/20260523_110707 (hg) ===
  [skip] X_train.npy missing

=== hierarchical/hg/runs/20260528_182149 (hg) ===
  [skip] X_train.npy missing

=== hierarchical/hg/runs/20260529_185811 (hg) ===
  K=5: wrote 5 PNGs -> cluster_centroids/k_5/
  K=6: wrote 6 PNGs -> cluster_centroids/k_6/
  K=7: wrote 7 PNGs -> cluster_centroids/k_7/
  K=8: wrote 8 PNGs -> cluster_centroids/k_8/
  K=9: wrote 9 PNGs -> cluster_centroids/k_9/
  K=10: wrote 10 PNGs -> cluster_centroids/k_10/
  K=11: wrote 11 PNGs -> cluster_centroids/k_11/
  K=12: wrote 12 PNGs -> cluster_centroids/k_12/
  K=13: wrote 13 PNGs -> cluster_centroids/k_13/
  K=14: wrote 14 PNGs -> cluster_centroids/k_14/
  K=15: wrote 15 PNGs -> cluster_centroids/k_15/
  K=16: wrote 16 PNGs -> cluster_centroids/k_16/
  K=17: wrote 17 PNGs -> cluster_centroids/k_17/
  K=18:

## Done

Commit + push the new PNGs:
```
git add 02_FBM_Clustering/outputs/clustering
git commit -m "214: per-K centroid backfill for raw / blob / minus101 / hg"
git push
```

Then MOBA's K slider will swap chip thumbnails on every feature set
(not just rawds).

**Sanity-check the rendering** before committing — open a sample of
the new PNGs and confirm they look right (raw / minus101 should be
129×300 heatmaps; blob and hg should be horizontal stripes).